In [0]:
from pyspark.sql.functions import current_timestamp, col, lit, to_timestamp, date_format, from_utc_timestamp
from delta.tables import DeltaTable

In [0]:
year = "2025"
month = "01"
batch_processing = f"{year}{str(month).zfill(2)}"

batch_processing

In [0]:
source_path = "/Volumes/oftalmo_sus/00_landing/raw_files/SIA_PA/*"
df_landing = spark.read.parquet(source_path)

In [0]:
df_bronze = (
    df_landing
    .withColumn("bronze_ingest_date", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("batch_processing", lit(batch_processing))
)

In [0]:
target_table = "oftalmo_sus.01_bronze.tb_datasus_sia_pa"

In [0]:
replace_condition = f"batch_processing = '{batch_processing}'"

(
    df_bronze.write
    .format("delta")
    .partitionBy("batch_processing") # Particionando pelo lote do arquivo
    .mode("overwrite")
    .option("replaceWhere", replace_condition) # Protegendo o resto do Data Lake
    .saveAsTable(target_table)
)

In [0]:
delta_table = DeltaTable.forName(spark, target_table)

history_df = delta_table.history()

display(
    history_df
    .orderBy(col("version").desc())
    .limit(1)
    .select(
        col("version"),
        date_format(from_utc_timestamp(col("timestamp"), "America/Sao_Paulo"), "dd-MM-yyyy HH:mm:ss").alias("timestamp"),
        col("operation"),
        col("operationMetrics.insert").alias("insert"),
        col("operationMetrics.delete").alias("delete"),
        col("operationMetrics.update").alias("update"),
        col("operationMetrics.numOutputRows").alias("num_rows")
    )
)